# Multi-Agent Orchestration 


## 전체 흐름

| 패턴 | 누가 다음 작업자를 결정하는가 | 특징 |
|---|---|---|
| 직접 만든 Sub-agent | Supervisor 에이전트 | 가장 단순하고 커스터마이즈 쉬움 |
| Supervisor | 중앙 supervisor | 역할 분배와 최종 종합 |
| Handoff | supervisor + worker | worker끼리 직접 위임 |
| Swarm | 현재 active agent | peer-to-peer 협업 |

예제 팀 구성:

| 역할 | 책임 |
|---|---|
| `market_researcher` | 시장 조사, 타깃 고객, 경쟁 구도 |
| `product_strategist` | 핵심 기능, 포지셔닝, MVP 범위 |
| `financial_analyst` | 수익 모델, 가격, 비용 구조 |
| `risk_reviewer` | 리스크, 실행 난이도, 검토 |

공통 프롬프트:

> AI 기반 영어 회화 앱의 MVP 기획안을 만들어줘. 타깃 고객, 핵심 기능, 수익 모델, 리스크를 포함해.


## 환경 준비


```text
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=mtvs2026-multi-agent
```


In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## 1. 직접 만든 Sub-agent


### 1.1. Worker 에이전트 2명 만들기


In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

llm = init_chat_model('openai:gpt-5.4-mini')

market_researcher = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 스타트업 시장 조사 담당자. 타깃 고객, 경쟁 구도, 시장 기회를 짧고 구체적으로 정리해.'
)

product_strategist = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 SaaS/앱 제품 전문가. MVP 핵심 기능과 포지셔닝을 명확하게 제안해'
)

### 1.2. Worker를 도구로 감싸기


In [3]:
@tool
def ask_market_researcher(prompt: str) -> str:
    """시장 규모, 타깃 고객, 경쟁 구도 조사가 필요할 때 호출, prompt에 자세한 요구사항을 넣어줘."""
    result = market_researcher.invoke({"messages": [HumanMessage(prompt)]})
    return result["messages"][-1].content


@tool
def ask_product_strategist(prompt: str) -> str:
    """제품 포지셔닝, MVP 기능, 차별화 전략이 필요할 때 호출"""
    result = product_strategist.invoke({"messages": [HumanMessage(prompt)]})
    return result["messages"][-1].content


### 1.3. Supervisor 에이전트


In [4]:
supervisor = create_agent(
    model=llm,
    tools=[ask_market_researcher, ask_product_strategist],
    system_prompt='''너는 스타트업 사업기획 리드.
    
    원칙:
    1. 사용자 요청을 받으면 어떤 전문가가 필요한지 먼저 판단
    2. 시장, 고객,경쟁은 ask_market_researcher에 위임
    3. MVP 기능, 포지셔닝은 ask_product_strategist에 위임
    4. 두 전문가의 답을 합쳐서 실행 가능한 MVP 기획안을 요약    
    '''
)


### 1.4. 복합 요청 실행


In [5]:
result = supervisor.invoke({
    "messages": [HumanMessage(
        "AI 기반 영어 회화 앱의 MVP 기획안을 만들어주세요."
        "타깃 고객, 핵심 기능, 포지셔닝을 포함해서."
    )]
})

print(result["messages"][-1].content)

아래는 두 관점(시장/고객/경쟁 + 제품 전략)을 합친 **AI 기반 영어 회화 앱 MVP 기획안**입니다.

---

# 1) MVP 한 줄 정의

**매일 10분, AI와 실전처럼 말하고 즉시 교정받는 영어 회화 코치 앱**

핵심은 “영어를 배우는 앱”보다 **“실제로 말하게 만드는 앱”**입니다.

---

# 2) 타깃 고객

## 1순위 타깃: 취업/이직 준비 20~34세
특히 아래 사용자에 맞습니다.
- 영어 면접 준비자
- 외국계/대기업/해외영업/PM/기획 직군 지원자
- 짧은 기간 안에 실전 영어가 필요한 사람

### 왜 이 타깃인가?
- 문제의식이 강해서 결제 이유가 분명함
- 사용 목적이 구체적이라 MVP 설계가 쉬움
- AI 면접 Q&A, 꼬리질문, 답변 리허설과 궁합이 좋음
- “합격”이라는 결과와 연결돼 지불 의사가 높음

## 보조 타깃
### 2. 직장인 실무 영어 사용자 25~45세
- 회의, 출장, 화상회의, 고객 응대 대비
- B2B 확장 가능성까지 고려 가능

### 3. 여행/유학/해외생활 준비자
- 생존 회화 중심
- 수요는 있으나 가격 민감도가 높아 MVP 1순위로는 약함

---

# 3) 포지셔닝

## 추천 포지셔닝
**“영어 면접과 실전 답변을 AI로 반복 연습하는 개인 코치”**

좀 더 넓게 잡으면:
**“내 상황에 맞는 영어를 매일 말하게 만드는 AI 회화 코치”**

### 포지셔닝 포인트
- 범용 영어 학습 앱이 아님
- 문법 앱도 아님
- 원어민 튜터 대체재도 아님
- **실전 회화 리허설 + 즉시 피드백**에 집중

### 메시지 예시
- “배우는 앱이 아니라, 말하게 만드는 앱”
- “면접, 회의, 출장 전에 먼저 AI와 연습하세요”
- “틀린 답변을 더 자연스럽게 바꿔주는 회화 코치”

---

# 4) 핵심 기능

MVP는 기능을 많이 넣는 것이 아니라, **회화 경험의 핵심 루프**를 잘 만드는 것이 중요합니다.

## 필수 기능 3개

### 1. AI 음성/텍스트 대화
- 사용자가 말하거

### 1.5. 왜 이 패턴이 좋은가

| 한 에이전트가 다 함 | Supervisor + Worker |
|---|---|
| 시장, 제품, 수익, 리스크를 한 프롬프트에 모두 넣음 | 역할별 worker 프롬프트를 작게 유지 |
| 조사, 기능 설계, 검토가 한 메시지 흐름에 모두 누적 | supervisor는 worker 결과만 받아 종합 |
| 토큰 비용과 역할 혼선 증가 | 토큰 절약과 역할 분리 |

직접 만든 Sub-agent 패턴은 구조가 단순해서 원리를 설명하기 좋습니다. 다만 worker 수가 늘어나면 도구 래퍼와 라우팅 규칙을 직접 관리해야 합니다.


## [실습]
1. `art_director` worker 등 추가 (이미지 프롬프트 생성 전문, 디자인 전문 등).
2. supervisor 가 어느 worker 를 부르는지 LangSmith trace 에서 확인.
3. Worker 에 도구를 각자 다르게 (lore_writer 에 wiki 검색, balance 에 spreadsheet 계산) 붙이기.


---

In [6]:
art_director = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 UI/UX 디자인 전문가. 사용자 경험을 고려한 디자인과 인터페이스를 제안해.'
)

@tool
def ask_art_director(prompt: str) -> str:
    """디자인, UI/UX 관련 질문이 필요할 때 호출"""
    result = art_director.invoke({"messages": [HumanMessage(prompt)]})
    return result["messages"][-1].content

supervisor_1 = create_agent(
    model=llm,
    tools=[ask_market_researcher, ask_product_strategist, ask_art_director],
    system_prompt='''너는 스타트업 사업기획 리드.
    
    원칙:
    1. 사용자 요청을 받으면 어떤 전문가가 필요한지 먼저 판단
    2. 시장, 고객, 경쟁은 ask_market_researcher에 위임
    3. MVP 기능, 포지셔닝은 ask_product_strategist에 위임
    4. 디자인, UI/UX는 ask_art_director 위임
    5. 세 전문가의 답을 합쳐서 실행 가능한 MVP 기획안을 요약    
    '''
)

result_1 = supervisor_1.invoke({
    "messages": [HumanMessage(
        "AI 기반 영상 편집 앱의 MVP 기획안을 만들어주세요."
        "타깃 고객, 핵심 기능, 포지셔닝을 포함해서."
    )]
})

print(result_1["messages"][-1].content)


좋습니다. 세 관점의 답을 합쳐서 **AI 기반 영상 편집 앱 MVP 기획안**으로 정리해드리겠습니다.

---

# 1) MVP 한 줄 정의

**“영상 편집을 배우지 않아도, 업로드만 하면 숏폼/릴스/홍보 영상을 몇 분 안에 완성해주는 AI 편집 앱”**

핵심은 편집 툴이 아니라 **결과물 생성기**로 포지셔닝하는 것입니다.

---

# 2) 타깃 고객

## 1순위: 개인 크리에이터 / 숏폼 제작자
- 유튜브 쇼츠, 인스타 릴스, 틱톡 운영자
- 영상 업로드 빈도가 높고 편집 시간이 가장 큰 부담
- MVP 가치 체감이 가장 빠름

**핵심 니즈**
- 자동 컷 편집
- 자동 자막
- 세로형 리프레임
- 훅 문구 추천
- 빠른 내보내기

## 2순위: 이커머스 셀러
- 스마트스토어, 쿠팡, 자사몰 운영자
- 상품 소개 영상이 반복적으로 필요
- 상품 1개로 여러 버전의 광고/숏폼을 만들어야 함

**핵심 니즈**
- 상품 이미지/설명 → 영상 자동 생성
- CTA 문구
- 채널별 비율 출력
- 반복 제작 효율

## 3순위: 소상공인 / 1인 사업자
- 카페, 네일샵, 헬스장, 지역 매장 운영자
- 홍보 영상 니즈는 높지만 편집 인력이 없음

**핵심 니즈**
- 이벤트/후기/매장 소개 영상 빠른 제작
- 템플릿 기반
- 저비용
- 쉬운 사용성

## 후순위: 마케터, 교육자
- 마케터는 유료 전환 가능성이 높지만 협업 기능이 필요해 MVP 범위를 넘길 수 있음
- 교육자는 하이라이트/요약 클립 니즈는 있으나 사용 빈도가 낮을 수 있음

---

# 3) 타깃 우선순위 추천

## MVP 초기 추천 시장
**개인 크리에이터 + 이커머스 셀러**

### 이유
- 반복 사용 빈도가 높음
- AI 자동화 효과가 즉시 보임
- 기능이 명확해서 제품 검증이 쉬움
- 유료 전환 테스트가 가능함

---

# 4) 포지셔닝

## 포지셔닝 문장
**“대본이나 원본 영상만 넣으면 AI가 자동으로 컷 편집, 자막, 훅 문구, 세로 비율 최적화까지 해주는 초간단 숏폼

## 2. Supervisor 패턴

- supervisor + worker 를 직접 손으로 만들어봤다면, **`langgraph-supervisor`** 라이브러리는 이 패턴을 한 줄로 묶어줍니다.

### 2.1. Worker 두 명 만들기


In [7]:
llm = init_chat_model("openai:gpt-5.4-mini")


@tool
def search_market(keyword: str) -> str:
    """스타트업 시장 조사 자료 검색. 데모용 가짜 데이터."""
    db = {
        "영어 회화": "성인 직장인과 취업 준비생의 회화 학습 수요가 높음. 기존 앱은 반복 학습과 실제 대화 지속률이 약점.",
        "AI 튜터": "개인화 피드백, 발음 교정, 상황극 대화가 핵심 차별화 포인트.",
        "경쟁 앱": "Duolingo, Speak, Cambly 등이 존재. 가격, 실시간성, 개인화 수준에서 차별화 필요.",
    }
    return db.get(keyword, "관련 시장 자료 없음")


@tool
def estimate_unit_economics(monthly_price: int, users: int, churn_rate: float) -> str:
    """월 구독 가격, 유료 사용자 수, 월 이탈률 기준 간단한 수익 모델 계산."""
    mrr = monthly_price * users
    retained_users = int(users * (1 - churn_rate))
    return f"MRR {mrr:,}원, 다음 달 예상 잔존 유저 {retained_users:,}명, 월 이탈률 {churn_rate:.1%}"

In [8]:
market_researcher = create_agent(
    model=llm,
    tools=[search_market],
    system_prompt='너는 스타트업 시장 조사 담당자. 타깃 고객, 경쟁 구도, 시장 기회를 짧고 구체적으로 정리해.',
    name='market_researcher'
)

product_strategist = create_agent(
    model=llm,
    tools=[],
    system_prompt='너는 제품 전문가. MVP 핵심 기능과 포지셔닝, 우선순위를 명확하게 제안해',
    name='produect_strategist'
)

financial_analyst = create_agent(
    model=llm,
    tools=[estimate_unit_economics],
    system_prompt='너는 재무 분석가. 수익 모델, 가격, 간단한 unit economics를 숫자로 정리해',
    name='financial_analyst'
)

### 2.2. Supervisor 한 줄로 만들기


In [9]:
%pip install langgraph_supervisor
from langgraph_supervisor import create_supervisor

supervisor_workflow = create_supervisor(
    agents=[market_researcher, product_strategist, financial_analyst],
    model=llm,
    prompt=(
        "너는 스타트업 사업기획 리드. 사용자 요청을 전문가에게 적절히 분배해.\n"
        "- 시장, 고객, 경쟁은 market_researcher \n"
        "- MVP 기능, 포지셔닝은 product_strategist \n"
        "- 수익 모델, 가격은 financial_analyst\n"
        "전문가 답을 종합해 실행 가능한 MVP 기획안을 작성."
    )
)

team = supervisor_workflow.compile()


  Using cached langgraph_supervisor-0.0.31-py3-none-any.whl.metadata (14 kB)
Using cached langgraph_supervisor-0.0.31-py3-none-any.whl (16 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2.3. 복합 요청 실행


In [10]:
result = team.invoke({
    'messages': [HumanMessage(
        "AI 기반 영어 회화 앱의 MVP 기획안을 만들어주세요."
        "타깃 고객, 핵심 기능, 수익 모델을 포함해서."
    )]
})

print(result['messages'][-1].content)

아래는 **AI 기반 영어 회화 앱 MVP 기획안**입니다.  
시장/고객/경쟁, 기능, 수익모델을 종합해 **바로 실행 가능한 형태**로 정리했습니다.

---

# 1) 제품 개요

**한 줄 정의**  
AI와 실전처럼 대화하며 영어 회화 실력을 키우는 모바일 앱

**핵심 가치**
- 언제 어디서나 말하기 연습 가능
- 틀려도 부담 없는 1:1 비공개 대화
- 즉시 수정 피드백으로 반복 학습 가능

---

# 2) 타깃 고객

## 1순위: 영어 말하기가 약한 성인 초중급자
- **연령:** 20~39세
- **특징:** 영어 공부 경험은 있으나 회화가 안 됨
- **니즈:**  
  - 일상에서 쓸 수 있는 말하기 연습
  - 문법보다 “실제로 말하는 감각” 회복
  - 짧고 자주 할 수 있는 학습 도구

## 2순위: 취업·이직 준비자 / 비즈니스 영어 수요층
- **특징:** 면접, 외국계 회사, 해외업무 대응 필요
- **니즈:**  
  - 자기소개/면접 답변 연습
  - 직무별 비즈니스 표현
  - 반복 가능한 모의 인터뷰

## 초반 제외 타깃
- 완전 초보자: 이탈률 높음
- 상급자: 만족 포인트 약함
- 아동/청소년: 안전/콘텐츠 요구가 큼

---

# 3) 핵심 기능 MVP

MVP는 **대화 → 피드백 → 복습** 3단계에 집중합니다.

## 필수 기능
### 1. AI 영어 대화
- 주제 선택형 대화
- 텍스트/음성 입력 지원
- 일상/여행/면접/비즈니스 등 기본 시나리오 제공
- 난이도 단계 선택 가능

### 2. 즉시 피드백
- 문법 수정
- 더 자연스러운 표현 제안
- 한국어로 짧은 설명 제공

### 3. 발음/말하기 연습
- 음성 인식 기반 응답
- 사용자가 말한 내용을 텍스트화
- 인식 결과와 간단한 발음 피드백 제공

### 4. 개인화 복습
- 자주 틀린 표현 저장
- 오늘의 복습 문장 자동 추천
- 이전 대화의 오답 재연습

### 5. 학습 기록
- 연습 시간
- 대화 횟수
- 자주 틀린 표현
- 진행도 대시

### 2.4. 누가 무엇을 답했는지 trace


In [11]:
print(f"총 메시지 수: {len(result['messages'])}\n")
for i, m in enumerate(result["messages"][-8:]):
    name = getattr(m, "name", None) or type(m).__name__
    content = (m.content if isinstance(m.content, str) else str(m.content))[:80]
    print(f"  [{i}] {name}: {content}")

총 메시지 수: 7

  [0] HumanMessage: AI 기반 영어 회화 앱의 MVP 기획안을 만들어주세요.타깃 고객, 핵심 기능, 수익 모델을 포함해서.
  [1] supervisor: 
  [2] transfer_to_market_researcher: Successfully transferred to market_researcher
  [3] market_researcher: 아래는 **AI 기반 영어 회화 앱 MVP 기획안**입니다.  
시장 관점에서 **가장 먼저 검증해야 할 타깃 고객, 꼭 필요한 기능, 초기에 
  [4] market_researcher: Transferring back to supervisor
  [5] transfer_back_to_supervisor: Successfully transferred back to supervisor
  [6] supervisor: 아래는 **AI 기반 영어 회화 앱 MVP 기획안**입니다.  
시장/고객/경쟁, 기능, 수익모델을 종합해 **바로 실행 가능한 형태**로 정리


### 2.5. Supervisor가 자동으로 하는 일

| 단계 | 동작 |
|---|---|
| 1 | 사용자 메시지를 받음 |
| 2 | 어떤 worker가 적합한지 판단(도구 호출 형식으로)  |
| 3 | 그 worker에게 메시지 전달 |
| 4 | worker 결과를 보고 추가 worker 호출 여부 결정 |
| 5 | 충분하면 최종 답변 합성 |

- `create_supervisor(agents=[...], model=llm, prompt=...)`는 직접 만든 Sub-agent 패턴의 반복 코드를 줄여줍니다. 
- worker마다 `name`이 필요하고, `.compile()` 후 일반 LangGraph처럼 `invoke()` 또는 `stream()`으로 실행합니다.


## [실습]
1. supervisor 가 worker 를 몇 번 호출했는지 messages 의 `name` 필드로 카운트.
2. LangSmith trace 에서 worker 별 토큰 비용 분리해서 보기.

---

## 3. Handoff 패턴

- Supervisor 패턴은 모든 결정이 중앙(supervisor)에 모입니다. 반대로 **Handoff** 는 한 에이전트가 "이건 너가 맡아" 하면서 직접 다른 에이전트에 넘기는 방식.
- `create_handoff_tool` 로 "넘기기 도구" 를 만들어 worker 끼리 주고받게 합니다.

### 3.1. Handoff 도구 만들기


In [12]:
from langgraph_supervisor import create_handoff_tool

# 각 worker에서 다른 worker로 넘기는 handoff 도구를 부여
handoff_to_market = create_handoff_tool(
    agent_name="market_researcher",
    description="시장 조사, 타깃 고객, 경쟁 분석이 필요할 때 호출."
)

handoff_to_product = create_handoff_tool(
    agent_name="product_strategist",
    description="MVP 기능, 포지셔닝, 제품 우선순위가 필요할 때 호출."
)

handoff_to_finance = create_handoff_tool(
    agent_name="finance_analyst",
    description="수익 모델, 가격, 비용 구조 계싼이 필요할 때 호출."
)

handoff_to_risk = create_handoff_tool(
    agent_name="risk_reviewer",
    description="기획안 검토, 리스크, 실행 난이도 평가가 필요할 때 호출."
)

### 3.2. 3명의 에이전트, 서로 위임 가능
시장 조사 담당자 ↔ 제품 전략가 ↔ 재무 분석가 ↔ 리스크 검토자가 서로에게 일을 넘길 수 있게 도구를 부여.

In [13]:
llm = init_chat_model("openai:gpt-5.4-mini")


@tool
def estimate_subscription_revenue(monthly_price: int, paid_users: int) -> str:
    """월 구독 가격과 유료 사용자 수로 MRR을 계산."""
    return f"예상 MRR: {monthly_price * paid_users:,}원"


market_researcher = create_agent(
    model=llm,
    tools=[handoff_to_product, handoff_to_finance, handoff_to_risk],
    system_prompt=(
        "너는 시장 조사 담당자. 타깃 고객과 경쟁 구도를 정리한 뒤 다음 단계가 필요한지 판단.\n"
        "- 제품 기능이 필요하면 product_strategist로 handoff\n"
        "- 수익 모델이 필요하면 financial_analyst로 handoff\n"
        "- 검토가 필요하면 risk_reviewer로 handoff"
    ),
    name="market_researcher",
)

product_strategist = create_agent(
    model=llm,
    tools=[handoff_to_market, handoff_to_finance, handoff_to_risk],
    system_prompt=(
        "너는 제품 전략가. MVP 기능과 포지셔닝을 정리한 뒤 다음 단계를 결정.\n"
        "- 시장 근거가 부족하면 market_researcher\n"
        "- 가격·수익 모델이 필요하면 financial_analyst\n"
        "- 리스크 검토가 필요하면 risk_reviewer"
    ),
    name="product_strategist",
)

financial_analyst = create_agent(
    model=llm,
    tools=[estimate_subscription_revenue, handoff_to_market, handoff_to_product, handoff_to_risk],
    system_prompt=(
        "너는 재무 분석가. 가격, 수익 모델, 비용 구조를 숫자로 정리.\n"
        "- 시장 가정이 부족하면 market_researcher\n"
        "- 기능 범위가 불명확하면 product_strategist\n"
        "- 검토가 필요하면 risk_reviewer"
    ),
    name="financial_analyst",
)

risk_reviewer = create_agent(
    model=llm,
    tools=[handoff_to_market, handoff_to_product, handoff_to_finance],
    system_prompt=(
        "너는 리스크 검토자. 기획안의 빈 곳과 실행 리스크를 짚고 부족하면 담당자에게 다시 handoff.\n"
        "충분하면 '검토 완료'라고 짧게 마무리."
    ),
    name="risk_reviewer",
)


### 3.3. Supervisor가 첫 진입을 정하고, 이후 worker handoff 보기

- Supervisor 가 `lore_writer` 로 첫 위임을 한 번 하고, 그 뒤로는 worker 내부의 handoff tool 이 다음 담당을 직접 지정합니다.


In [14]:
from langgraph_supervisor import create_supervisor

workflow = create_supervisor(
    agents=[market_researcher, product_strategist, financial_analyst, risk_reviewer],
    model=llm,
    prompt="첫 진입자는 market_researcher. 이후 worker들이 필요한 전문가에게 직접 넘기게 둬."
)

team = workflow.compile()

### 3.4. 실행, 누가 누구에게 넘기는지 보기

- LLM 이 서로에게만 넘기는 무한 핑퐁을 막기 위해 `recursion_limit` 을 함께 지정합니다.


In [15]:
result = team.invoke({
    "messages":[HumanMessage(
        "AI 기반 영어 회화 앱의 MVP 기획안을 만들어줘."
        "타깃 고객, 핵심 기능, 수익 모델, 리스크를 포함해."
    )]
})

print(result['messages'][-1].content)

Ignoring unknown node name finance_analyst in pending sends


아래는 **AI 기반 영어 회화 앱 MVP 기획안**입니다.  
시장 관점에서 **타깃 고객, 핵심 기능, 수익 모델, 리스크** 중심으로 실용적으로 정리했습니다.

---

# AI 기반 영어 회화 앱 MVP 기획안

## 1) 제품 한 줄 정의
**“사용자가 부담 없이 매일 10분, AI와 실제 회화처럼 영어를 말하고 바로 피드백받는 앱”**

핵심은  
- **말하기 중심**
- **즉시 피드백**
- **짧고 반복 가능한 학습**
- **실제 상황 기반 대화**

입니다.

---

## 2) 타깃 고객

### 1차 타깃
**영어 말하기에 가장 큰 심리적 장벽을 가진 학습자**
- 한국 20~40대 직장인, 대학생
- 영어 공부는 했지만 말하기가 어려운 사람
- 학원/전화영어는 비용·시간 부담이 큰 사람
- 틀리는 것에 대한 두려움이 큰 사람

### 2차 타깃
**실용 영어가 필요한 사용자**
- 해외여행 준비자
- 해외 취업/이직 준비자
- 외국인 동료와 일하는 직장인
- 면접·발표·미팅 대비가 필요한 사용자

### 사용자 페르소나 예시
- **직장인 A(31세)**: 회화 학원 갈 시간 없음, 퇴근 후 10분만 투자 가능
- **대학생 B(24세)**: 토익 점수는 있지만 실제 말하기가 안 됨
- **여행 준비 C(28세)**: 출국 전 2주 안에 실전 표현 익히고 싶음

---

## 3) 문제 정의

기존 영어 학습 제품의 주요 문제:
1. **말할 기회가 부족함**
2. **사람 앞에서 말하는 부담이 큼**
3. **피드백이 늦거나 형식적임**
4. **학습이 지루해서 지속이 어려움**
5. **상황별 표현을 실전에 바로 쓰기 어려움**

MVP는 이 문제를 해결하는 데 집중해야 합니다.

---

## 4) MVP 핵심 기능

MVP는 “많이 넣는 것”보다 **꾸준히 쓰게 만드는 최소 기능**이 중요합니다.

### A. AI 대화 기능
- 사용자가 선택한 상황에서 AI와 영어로 대화
- 예: 자기소개, 카페 주문, 여행 체크인, 면접,

### 3.5. Supervisor vs Handoff, 언제 무엇을 쓰는가

| 상황 | 권장 |
|---|---|
| 작업 흐름이 예측 가능하고 중앙 통제가 필요함(분기 적음) | Supervisor |
| worker끼리 다음 담당자를 동적으로 정해야 함(누가 다음인지 case-by-case) | Handoff |
| 단순 분배와 최종 종합이 중요함 | Supervisor |
| 전문가들이 서로 핑퐁하며 협의해야 함 | Handoff |

- Handoff는 유연하지만 무한 위임이 생길 수 있습니다. `recursion_limit`을 항상 함께 보여주는 편이 안전합니다.


## [실습]

1. handoff 도구의 description 을 더 엄격하게 적고 호출 빈도 변화 관찰.
2. recursion_limit 을 5 로 낮춰 무한 핑퐁 방지 (LLM 이 서로에게만 넘기는 경우).
3. handoff 시점에 LangSmith metadata 에 "reason" 을 남기도록.


---

## 4. Swarm 패턴
- `langgraph-swarm` 은 supervisor 없이 "누가 활성 에이전트인가" 를 동적으로 바꿔가며 진행. peer-to-peer 협업이 필요할 때.

### 4.1. Worker들, Swarm용 handoff 도구

- Swarm에서는 현재 활성 에이전트가 다른 에이전트로 직접 handoff합니다. 이 예제에서는 일반 도구 호출과 handoff 도구 호출이 한 응답에서 섞여 OpenAI 메시지 형식 오류가 나지 않도록 Swarm 전용 LLM에서 병렬 도구 호출을 끕니다.


In [16]:
%pip install langgraph_swarm
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_openai import ChatOpenAI
from langgraph_swarm import create_swarm, create_handoff_tool as create_swarm_handoff_tool

swarm_llm: BaseChatModel = ChatOpenAI(
    model='gpt-5.4-mini',
    model_kwargs={'parallel_tool_calls': False}
)

swarm_to_market = create_swarm_handoff_tool(
    agent_name="swarm_market_researcher",
    description="시장 조사 담당자에게 넘김."
)

swarm_to_product = create_swarm_handoff_tool(
    agent_name="swarm_product_strategist",
    description="제품 전략가에게 넘김."
)

swarm_to_finance = create_swarm_handoff_tool(
    agent_name="swarm_financial_analyst",
    description="재무 분석가에게 넘김."
)

swarm_to_risk = create_swarm_handoff_tool(
    agent_name="swarm_risk_reviewer",
    description="리스크 검토자에게 넘김."
)

  Using cached langgraph_swarm-0.1.0-py3-none-any.whl.metadata (10 kB)
Using cached langgraph_swarm-0.1.0-py3-none-any.whl (10 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
@tool
def estimate_mrr(monthly_price: int, paid_users: int) -> str:
    """월 구독 가격과 유료 사용자 수로 MRR을 계산."""
    return f"MRR = {monthly_price * paid_users:,}원"

In [18]:
swarm_market_researcher = create_agent(
    model=swarm_llm,
    tools=[swarm_to_product, swarm_to_finance, swarm_to_risk],
    system_prompt=(
        "너는 시장 조사 담당자. 타깃 고객과 경쟁 구도를 정리해. "
        "제품 전략이 필요하면 swarm_product_strategist에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_market_researcher",
)

swarm_product_strategist = create_agent(
    model=swarm_llm,
    tools=[swarm_to_market, swarm_to_finance, swarm_to_risk],
    system_prompt=(
        "너는 제품 전략가. 핵심 기능과 포지셔닝을 정리해. "
        "수익 모델이 필요하면 swarm_financial_analyst에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_product_strategist",
)

swarm_financial_analyst = create_agent(
    model=swarm_llm,
    tools=[estimate_mrr, swarm_to_market, swarm_to_product, swarm_to_risk],
    system_prompt=(
        "너는 재무 분석가. 가격과 수익 모델을 숫자로 정리해. "
        "검토가 필요하면 swarm_risk_reviewer에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_financial_analyst",
)

swarm_risk_reviewer = create_agent(
    model=swarm_llm,
    tools=[swarm_to_market, swarm_to_product, swarm_to_finance],
    system_prompt=(
        "너는 리스크 검토자. 타깃 고객, 기능, 수익 모델, 리스크가 모두 있으면 '검토 완료'로 마무리해. "
        "부족한 부분이 있으면 해당 담당자에게 넘겨. "
        "한 번에 하나의 도구만 호출해."
    ),
    name="swarm_risk_reviewer",
)

### 4.2. Swarm 만들기, 기본 활성 에이전트 지정


In [19]:
swarm_workflow = create_swarm(
    agents=[
        swarm_market_researcher,
        swarm_product_strategist,
        swarm_financial_analyst,
        swarm_risk_reviewer        
    ],
    default_active_agent="swarm_market_researcher"
)

# 멀티턴에서 마지막 active_agent를 이어가려면 checkpointer를 추가함
# from langgraph.checkpoint.memory import InMemorySaver
# swarm = swarm_workflow.compile(checkpointer=InMemorySaver())

swarm = swarm_workflow.compile()

### 4.3. 실행


In [20]:
result = swarm.invoke(
    {
        "messages":[
            HumanMessage(
                "기업 임직원용 AI 영어 회화 코칭 서비스를 만들려고 한다. "
                "B2B 타깃 고객, 핵심 기능, 가격 정책, 도입 리스크를 정리해줘."
            )
        ]
    },
    config={"recursion_limit":18}
)

In [21]:
print("=== Active agent 시퀀스 ===")
for message in result["messages"]:
    name = getattr(message, "name", None)
    if name:
        print(f"  -> {name}")

print("\n마지막 active_agent:", result.get("active_agent"))
print("\n최종:", result["messages"][-1].content[:300])

=== Active agent 시퀀스 ===
  -> swarm_market_researcher
  -> transfer_to_swarm_product_strategist
  -> swarm_product_strategist
  -> transfer_to_swarm_financial_analyst
  -> swarm_financial_analyst

마지막 active_agent: swarm_financial_analyst

최종: 기업 임직원용 AI 영어 회화 코칭 서비스는 B2B 관점에서 **“교육 효과가 측정되는 업무형 언어 코칭”**으로 포지셔닝하는 것이 좋습니다. 아래처럼 정리할 수 있습니다.

## 1) B2B 타깃 고객

### 1순위 타깃
- **해외 영업/사업개발 조직**
  - 영어 미팅, 피치, 협상, 이메일 대응 빈도가 높음
- **글로벌 협업 조직**
  - 외국계 본사/지사와 정기 회의가 많은 팀
- **해외 고객 대응 부서**
  - CS, 기술지원, PM, 운영 조직
- **대기업/중견기업 HRD·L&D 부서**
  - 사내 교육 


### 4.4. Supervisor / Handoff / Swarm 한눈에

| 패턴 | 누가 결정 | 라이브러리 |
|---|---|---|
| Supervisor | supervisor 한 명 | `langgraph-supervisor` |
| Supervisor + handoff 도구 | supervisor + worker | `langgraph-supervisor` |
| Swarm | 활성 에이전트 본인 | `langgraph-swarm` |

- Swarm은 supervisor가 없고 **어느 에이전트가 활성인지**를 state로 관리합니다.
- `default_active_agent`가 시작점이고, `active_agent`가 다음 턴의 시작점을 결정합니다. 현재 예제는 일반 도구와 handoff 도구가 동시에 호출되어 메시지 짝이 깨지는 문제를 피하기 위해 `parallel_tool_calls=False`를 사용합니다.


### 정리

- Sub-agent는 자식 에이전트를 도구처럼 호출해 역할과 컨텍스트를 분리합니다.
- 직접 만든 Sub-agent 패턴은 원리를 설명하기 좋지만, worker가 늘어나면 래퍼와 라우팅 관리가 복잡해집니다.
- `langgraph-supervisor`는 중앙 사업기획 리드가 worker 호출과 최종 종합을 맡는 구조에 적합합니다.
- Handoff는 worker끼리 다음 담당자를 직접 정해야 하는 동적 협업에 유용합니다.
- Swarm은 supervisor 없이 active agent가 이동하는 peer-to-peer 구조입니다.
- 동적 협업 구조는 무한 핑퐁이 생길 수 있으므로 `recursion_limit`과 trace 확인이 중요합니다.


### [실습]

1. handoff 도구 description 을 더 좁히면 호출 패턴이 어떻게 변하는지.
2. 활성 에이전트가 무한 핑퐁할 때 `recursion_limit` 으로 멈추기.
3. swarm 의 state 에 마지막 active_agent 가 어떻게 저장되는지 확인 (`result.get("active_agent")`).
